# 🚀 Research Paper Summarizer - Colab Fine-Tuning Pipeline
This notebook contains the complete pipeline to train the BART model on a high-end cloud GPU for free using Google Colab.

### ⚠️ Prerequisites:
1. On the left sidebar, click the **Folder** icon.
2. Upload your `inputs.json` and `outputs - Copy.json` files.
3. At the top right, click **Connect**. Then go to **Runtime > Change runtime type** and ensure **T4 GPU** is selected.

## 1. Install Dependencies

In [3]:
!pip install -q transformers evaluate rouge_score datasets PyPDF2 pyttsx3 accelerate
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 26.1 MB/s eta 0:00:00


True

## 2. Load and Preprocess Data

In [2]:
import json
import datasets
from transformers import AutoTokenizer

MODEL_NAME = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_and_match_data(inputs_file, outputs_file):
    with open(inputs_file, 'r', encoding='utf-8') as f:
        inputs_data = json.load(f)
    with open(outputs_file, 'r', encoding='utf-8') as f:
        outputs_data = json.load(f)

    # Map inputs using 'id' as the key
    # Use None instead of null for Python
    inputs_dict = {str(item.get('id')): item.get('full_text', '') for item in inputs_data if item.get('id') is not None}
    matches = []

    for out_item in outputs_data:
        # Match using the 'id' field
        item_id = str(out_item.get("id", ""))
        if item_id in inputs_dict:
            combined_summary = (f"SUMMARY: {out_item.get('summary', '')}\n"
                                f"METHODOLOGY: {out_item.get('methodology', '')}\n"
                                f"FINDINGS: {out_item.get('findings', '')}\n"
                                f"CONCLUSION: {out_item.get('conclusion', '')}\n"
                                f"LIMITATIONS: {out_item.get('limitations', '')}")
            matches.append({
                "input_text": f"Summarize the following research paper:\n\n{inputs_dict[item_id]}",
                "target_text": combined_summary
            })

    print(f"Matched {len(matches)} pairs using ID!")
    if len(matches) == 0:
        raise ValueError("No matching IDs found between inputs and outputs. Please ensure both JSON files contain a matching 'id' field.")

    return datasets.Dataset.from_list(matches)

# Re-run loading logic
try:
    df = load_and_match_data("inputs.json", "outputs - Copy.json")

    if len(df) > 1:
        df_split = df.train_test_split(test_size=0.15, seed=42)
        train_dataset = df_split['train']
        val_dataset = df_split['test']
    else:
        print("Warning: Dataset too small to split. Using full dataset for both train and validation.")
        train_dataset = df
        val_dataset = df

    def tokenize_fn(batch):
        model_inputs = tokenizer(batch['input_text'], max_length=1024, truncation=True, padding='max_length')
        labels = tokenizer(batch['target_text'], max_length=512, truncation=True, padding='max_length')
        model_inputs['labels'] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label]
            for label in labels['input_ids']
        ]
        return model_inputs

    train_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=['input_text', 'target_text'])
    val_dataset = val_dataset.map(tokenize_fn, batched=True, remove_columns=['input_text', 'target_text'])
    print("Tokenization Complete!")
except Exception as e:
    print(f"❌ Error: {e}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Matched 88 pairs using ID!


Map:   0%|          | 0/74 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Tokenization Complete!


## 3. Define Evaluation Metrics (ROUGE)

In [4]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    result = rouge.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    result = {key: float(value) * 100 for key, value in result.items() if isinstance(value, (int, float, np.number))}
    return {k: round(v, 4) for k, v in result.items()}

## 4. Fine-Tune the Model
*This uses the GPU to train much faster than a local PC.*

In [5]:
from transformers import AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
import torch

# Ensure MODEL_NAME is defined (it's usually defined in the preprocessing cell)
MODEL_NAME = "facebook/bart-large-cnn"

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart-research-finetuned",
    predict_with_generate=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=2,  # Larger batch size possible on Colab T4 GPU
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,  # Effective batch = 8
    weight_decay=0.1,
    save_total_limit=2,
    num_train_epochs=5,             # 5 Epochs for better learning
    fp16=True,                      # Ultra-fast mixed precision on GPU
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss"
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss,Rouge1,Rouge2,Rougel,Rougelsum
1,No log,4.383882,10.459700,0.734000,7.337700,9.928200
2,No log,4.061730,18.929900,1.899900,10.999800,17.735200
3,No log,3.911695,22.185000,3.024400,12.069400,21.575800
4,No log,3.867882,19.320900,2.488900,10.840200,18.545600
5,No log,3.842520,19.591900,2.019300,10.791100,18.580000


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


TrainOutput(global_step=25, training_loss=28.7914404296875, metrics={'train_runtime': 859.8745, 'train_samples_per_second': 0.43, 'train_steps_per_second': 0.029, 'total_flos': 801828702781440.0, 'train_loss': 28.7914404296875, 'epoch': 5.0})

## 5. Evaluate and Export Model
Save the model and run evaluation to see your new ROUGE scores.

In [6]:
from transformers import AutoTokenizer

# Ensure tokenizer is defined for saving
MODEL_NAME = "facebook/bart-large-cnn"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Final Evaluation Scores:")
print(trainer.evaluate())

# Save model locally in Colab
trainer.save_model("./final_model")
tokenizer.save_pretrained("./final_model")

# Zip it up to download
!zip -r final_model.zip ./final_model
print("\n\u2705 Done! You can now download final_model.zip from the filesystem on the left.")

Final Evaluation Scores:


{'eval_loss': 3.842519760131836, 'eval_rouge1': 19.5919, 'eval_rouge2': 2.0193, 'eval_rougeL': 10.7911, 'eval_rougeLsum': 18.58, 'eval_runtime': 27.7411, 'eval_samples_per_second': 0.505, 'eval_steps_per_second': 0.252, 'epoch': 5.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: final_model/ (stored 0%)
  adding: final_model/generation_config.json (deflated 59%)
  adding: final_model/training_args.bin (deflated 53%)
  adding: final_model/tokenizer.json (deflated 82%)
  adding: final_model/model.safetensors (deflated 7%)
  adding: final_model/config.json (deflated 59%)
  adding: final_model/tokenizer_config.json (deflated 50%)

✅ Done! You can now download final_model.zip from the filesystem on the left.


In [17]:
import torch
import PyPDF2
import re
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from google.colab import files

# 1. Load your fine-tuned model and tokenizer
model_path = "./final_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to("cuda")

def summarize_pdf_structured(pdf_path):
    # 2. Extract text from the PDF
    text = ""
    try:
        with open(pdf_path, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            for page in reader.pages[:6]:
                content = page.extract_text()
                if content:
                    text += content + "\n"
    except Exception as e:
        return f"Error reading PDF: {e}"

    if not text.strip():
        return "Could not extract text from PDF. Ensure it is not a scanned image."

    # 3. Preprocess and Tokenize - Modified to ask for structured output
    input_prompt = f"Provide a structured summary for the following research paper, including sections for SUMMARY, METHODOLOGY, FINDINGS, CONCLUSION, and LIMITATIONS:\n\n{text}"
    inputs = tokenizer(input_prompt, max_length=1024, truncation=True, return_tensors="pt").to("cuda")

    # 4. Generate summary
    summary_ids = model.generate(
        inputs["input_ids"],
        num_beams=5,
        max_length=800,
        min_length=250,
        no_repeat_ngram_size=3,
        length_penalty=1.5,
        early_stopping=True
    )

    # 5. Decode the raw output
    raw_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

    # 6. Improved Regex to split and clean headers
    headers = ["SUMMARY:", "METHODOLOGY:", "FINDINGS:", "CONCLUSION:", "LIMITATIONS:"]
    # This regex looks for the header names regardless of extra spaces or case
    pattern = r'(SUMMARY:|METHODOLOGY:|FINDINGS:|CONCLUSION:|LIMITATIONS:)'
    sections = re.split(pattern, raw_summary, flags=re.IGNORECASE)

    # Create a clean dictionary or display string
    structured_data = {}
    for i in range(1, len(sections), 2):
        header = sections[i].upper().strip()
        content = sections[i+1].strip() if i+1 < len(sections) else "No data generated for this section."
        structured_data[header] = content

    # 7. Display them separately
    final_display = ""
    for header, content in structured_data.items():
        final_display += f"### {header}\n{content}\n\n---\n"

    return final_display if final_display else raw_summary

# 8. Call the function outside after uploading the PDF
print("🚀 Please upload the research paper (PDF):")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n{'='*50}\n📄 ANALYZING: {filename}\n{'='*50}")
    structured_summary = summarize_pdf_structured(filename)
    print("\n✨ GENERATED ANALYSIS:\n")
    print(structured_summary)
    print(f"{'='*50}")

Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

🚀 Please upload the research paper (PDF):


Saving final research paper.pdf to final research paper (1).pdf

📄 ANALYZING: final research paper (1).pdf

✨ GENERATED ANALYSIS:

### SUMMARY:
This paper presents a deep learning model, a “Convolutional Neural Network -Long Short -Term Memory (CNN -LSTM), in order to forec ast 5 minutes power demand. The model is trained on a detailed, four year multivariable dataset, combining power demand with the six important features in the dataset. The improved model showed good performance on the test data, attaining a R -squared of 0.9520, a Mean absolute  percentage error (MAPE) of 4.66%, a Root mean squared (RMSE) of 302.21 Kwatts and Mean Absolute Error (MAE)  of 199.17 Kwatts. The results prove the model’s ability to capture complicated spatiotemporal dependencies, offering a strong and generalizable solution for granular electric ity forecasting. The main used model was the CNN -L STM hybrid, which combines the power of 2:1 Convolutional Network (CNN) with lagged features (Neural Network)